In [1]:
# Instalação e configuração no Google Colab
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium beautifulsoup4 > /dev/null

# Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Configurações do navegador
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# URL da categoria
url = "https://www.telhanorte.com.br/pisos-e-revestimentos/porcelanatos"
driver.get(url)
time.sleep(5)

# Clicar no botão "Mostrar mais" até que ele desapareça
produtos_anteriores = -1

while True:
    time.sleep(3)
    produtos_elementos = driver.find_elements(By.CSS_SELECTOR, "div.vtex-search-result-3-x-galleryItem")
    total_produtos = len(produtos_elementos)

    try:
        botao_mais = driver.find_element(By.CSS_SELECTOR, "a.vtex-button[rel='next']")
        driver.execute_script("arguments[0].scrollIntoView(true);", botao_mais)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", botao_mais)
        print(f"Carregando mais produtos... Total até agora: {total_produtos}")
    except NoSuchElementException:
        print("Botão 'Mostrar mais' não encontrado. Finalizando.")
        break
    except ElementClickInterceptedException:
        print("Não foi possível clicar no botão. Finalizando.")
        break

    if total_produtos == produtos_anteriores:
        print("Nenhum novo produto carregado. Finalizando.")
        break
    produtos_anteriores = total_produtos

# Coletar links dos produtos
produtos_elementos = driver.find_elements(By.CSS_SELECTOR, "div.vtex-search-result-3-x-galleryItem")
links = []

for produto in produtos_elementos:
    try:
        link = produto.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
        if link:
            links.append(link)
    except:
        continue

driver.quit()

# Função para extrair os detalhes de cada produto
def extrair_detalhes_produto(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None
        soup = BeautifulSoup(response.text, "html.parser")

        nome = soup.find("h1", class_="vtex-store-components-3-x-productNameContainer")
        nome = nome.text.strip() if nome else "N/A"

        codigo = soup.find("span", class_="vtex-product-identifier-0-x-product-identifier__value")
        codigo = codigo.text.strip() if codigo else "0"

        preco_original = soup.select_one(".vtex-product-price-1-x-listPriceWithUnitMultiplier strike")
        preco_original = preco_original.text.strip() if preco_original else "N/A"

        preco_promocional = soup.select_one(".telhanorte-telha-store-app-1-x-preco-best-un")
        preco_promocional = preco_promocional.text.strip() if preco_promocional else "N/A"

        preco_m2 = soup.select_one(".telhanorte-telha-store-app-1-x-preco-m2")
        preco_m2 = preco_m2.text.strip() if preco_m2 else "N/A"

        imagem = soup.find("img", class_="telhanorte-telha-store-app-1-x-imagem-destaque")
        imagem = imagem['src'] if imagem else "N/A"

        return [nome, codigo, preco_original, preco_promocional, preco_m2, imagem, url]
    except:
        return None

# Coletar dados detalhados de cada produto
dados_produtos = []

for i, link in enumerate(links):
    print(f"[{i+1}/{len(links)}] Coletando: {link}")
    detalhes = extrair_detalhes_produto(link)
    if detalhes:
        dados_produtos.append(detalhes)

# Salvar em CSV
df = pd.DataFrame(dados_produtos, columns=[
    "Nome", "Código", "Preço Original", "Preço Promocional", "Preço por m²", "Imagem", "URL"
])
df.to_csv("telhanorte_porcelanatos_detalhado.csv", index=False)
df.head()


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
Carregando mais produtos... Total até agora: 40
Carregando mais produtos... Total até agora: 80
Carregando mais produtos... Total até agora: 120
Carregando mais produtos... Total até agora: 160
Carregando mais produtos... Total até agora: 200
Carregando mais produtos... Total até agora: 240
Carregando mais produtos... Total até agora: 280
Carregando mais produtos... Total até agora: 320
Carregando mais produtos... Total até agora: 360
Carregando mais produtos... Total até agora: 400
Carregando mais produtos... Total até agora: 440
Carregando mais produtos... Total até agora: 480
Carregando mais produtos... Total até agora: 520
Carregando mais produtos... Total até agora: 560
Carregando mais produtos... Total 

,Nome,Código,Preço Original,Preço Promocional,Preço por m²,Imagem,URL
0,Porcelanato Nord Ris 90X90Cm Natural Retificad...,2547910,N/A,"R$ 250,00","R$ 103,31/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-nord...
1,Porcelanato Confete White Natural 100x100cm Re...,1751271,N/A,"R$ 300,61","R$ 150,31/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-conf...
2,Porcelanato Mont Blanc Acetinado 90X90Cm Ret C...,2518449,N/A,"R$ 181,28","R$ 111,90/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-mont...
3,Porcelanato Cimentício Acetinado Borda Reta In...,2363550,N/A,"R$ 130,08","R$ 59,13/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-esma...
4,"Porcelanato Caminho Sgr Natural 87,7X87,7Cm Re...",2515032,N/A,"R$ 246,25","R$ 159,90/m²",https://telhanorte.vtexassets.com/arquivos/ids...,https://www.telhanorte.com.br/porcelanato-cami...
